In [ ]:
import gc
import glob
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
from collections import deque
from contextlib import nullcontext
from functools import partial
from typing import List, Optional, Tuple
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import GPUtil
import colorama
import numpy as np
import torch
from torch.autograd.profiler import record_function
from torch.utils.data import DataLoader
import pyiqa
import dist
from utils import arg_util, misc
from utils.data import build_dataset, pil_load
from utils.data_sampler import DistInfiniteBatchSampler
from models.vqvae import VQVAE
from utils.arg_util import Args
from PIL import Image
import matplotlib.pyplot as plt
import dist
import torch.distributed as tdist
from torchvision.transforms import InterpolationMode, transforms
from utils.data_loader import DIV2KData


args = Args()
# args.data = "./data/df2k_ost/GT_resized"
# args.data = "./data/DIV2K_train_HR"
# args.data = "./data/train"
# args.data = "./data/brats_256_t1_new/train"


In [ ]:
def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    tdist.init_process_group("nccl", rank=rank, world_size=world_size)
setup(0,1)

In [ ]:
def normalize_01_into_pm1(x):  # normalize x from [0, 1] to [-1, 1] by (x*2) - 1
    return x.add(x).add_(-1)
args.data = '/home/why/vaex/data/brats_256_t2_2021_pair_png_with_ref/val'
val_aug = [
        transforms.ToTensor(), normalize_01_into_pm1,
    ]
val_aug = transforms.Compose(val_aug)
val_set = DIV2KData(data_dir=args.data, transform=val_aug, augment=False)  # todo: junfeng; only `train_set` required, no need to create a 'validation_set'
ld_val = DataLoader(
    dataset=val_set, num_workers=args.workers, pin_memory=True,shuffle=True
)


In [ ]:

# vae_ckpt = f"ckpt_vaex/brats256-new/ckpt-150.pth"
vae_ckpt = f"local_output/ckpt-last.pth"
# vae_ckpt = f"local_output/ckpt-900.pth"
# vae_ckpt = f'vae_ch160v4096z32.pth'

args.vocab_size = 4096
args.vocab_width = 32
# args.vocab_size = 4096

vae = VQVAE(vocab_size=args.vocab_size, z_channels=args.vocab_width, ch=args.ch, 
            test_mode=True, share_quant_resi=args.share_quant_resi, v_patch_nums=args.patch_nums).to(args.device).eval()
# print(torch.load(vae_ckpt, map_location='cpu')['trainer']['vae_wo_ddp'])
vae.load_state_dict(torch.load(vae_ckpt, map_location='cpu')['trainer']['vae_wo_ddp'])
# vae.load_state_dict(torch.load(vae_ckpt, map_location='cpu'))

# x1 = torch.load(vae_ckpt, map_location='cpu')['trainer']['vae_wo_ddp']
# x2 = torch.load(vae_ckpt_init, map_location='cpu')
# for keys in x1.keys():
#     print(keys,x1[keys].mean() - x2[keys].mean())


In [ ]:
def process_image(x):
    """处理单张图片"""    
    x = x.clamp(-1.0, 1.0)
    x = x[0].detach().cpu().permute(1, 2, 0).numpy()  # 转换为 HWC 格式的 numpy 数组
    x = (x * 0.5 + 0.5) * 255  # 反归一化并缩放到 [0, 255]
    x = x.astype(np.uint8)  # 转换为 uint8
    return x
def img2tensor(img):
    img = (img / 255.).astype('float32')
    if img.ndim ==2:
        img = np.expand_dims(np.expand_dims(img, axis = 0),axis=0)
    else:
        img = np.transpose(img, (2, 0, 1))  # C, H, W
        img = np.expand_dims(img, axis=0)
    img = np.ascontiguousarray(img, dtype=np.float32)
    tensor = torch.from_numpy(img)
    return tensor
def rgb2ycbcr_pt(img, y_only=False):
    """Convert RGB images to YCbCr images (PyTorch version).
    It implements the ITU-R BT.601 conversion for standard-definition television. See more details in
    https://en.wikipedia.org/wiki/YCbCr#ITU-R_BT.601_conversion.
    Args:
        img (Tensor): Images with shape (n, 3, h, w), the range [0, 1], float, RGB format.
         y_only (bool): Whether to only return Y channel. Default: False.
    Returns:
        (Tensor): converted images with the shape (n, 3/1, h, w), the range [0, 1], float.
    """
    if y_only:
        weight = torch.tensor([[65.481], [128.553], [24.966]]).to(img)
        out_img = torch.matmul(img.permute(0, 2, 3, 1), weight).permute(0, 3, 1, 2) + 16.0
    else:
        weight = torch.tensor([[65.481, -37.797, 112.0], [128.553, -74.203, -93.786], [24.966, 112.0, -18.214]]).to(img)
        bias = torch.tensor([16, 128, 128]).view(1, 3, 1, 1).to(img)
        out_img = torch.matmul(img.permute(0, 2, 3, 1), weight).permute(0, 3, 1, 2) + bias

    out_img = out_img / 255.
    return out_img
psnr_metric = pyiqa.create_metric('psnr', device="cpu")
ssim_metric = pyiqa.create_metric('ssim', device="cpu")

tot = 0
for data in ld_val:
    with torch.no_grad():
        data = data.to(args.device)
        
        rec_B3HW, usage , Lq  = vae(data)
        vaex_first_rec = vae.decoder(vae.post_quant_conv(vae.quant_conv(vae.encoder(data))))
        
        rec_B3HW = process_image(rec_B3HW)
        data_ = process_image(data)
        vaex_first_rec = process_image(vaex_first_rec)
        
        gt_ = rgb2ycbcr_pt(img2tensor(data_),  y_only=True).to(torch.float64)
        vaex_predict = rgb2ycbcr_pt(img2tensor(rec_B3HW),  y_only=True).to(torch.float64)
        vaex_first_predict = rgb2ycbcr_pt(img2tensor(vaex_first_rec),  y_only=True).to(torch.float64)
        print("vaex predict and gt", psnr_metric(vaex_predict, gt_))
        print("vaex first predict and gt", psnr_metric(vaex_first_predict, gt_))
        
        plt.figure()
        plt.subplot(1, 2, 1)
        plt.imshow(rec_B3HW)
        plt.subplot(1, 2, 2)
        plt.imshow(data_)
        plt.show()
        
        predict_idx = vae.img_to_idxBl(data)
        nup_predict = vae.idxBl_to_img(predict_idx, same_shape=True, last_one=False)
        predict_combined = np.concatenate([process_image(x) for x in nup_predict], axis=1)
        plt.imshow(predict_combined.astype(np.uint8))
        plt.show()
        
    tot = tot + 1
    if tot > 10:
        break
